# 01. SECOM Data Loading

## Project: Semiconductor Yield Prediction

### Objective
This notebook loads and inspects the SECOM semiconductor manufacturing dataset.

### Goals
1. Verify the Python analysis environment.
2. Load the raw SECOM dataset.
3. Examine the dataset structure.
4. Inspect the Pass/Fail target distribution.
5. Identify missing-value and data-quality issues before preprocessing.

In [1]:
import sys
import platform

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

print("\nOperating system:")
print(platform.platform())

Python version:
3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]

Python executable:
C:\Users\ark34\AppData\Local\Programs\Python\Python314\python.exe

Operating system:
Windows-11-10.0.26200-SP0


In [2]:
packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "sklearn",
    "imblearn",
    "xgboost"
]

for package in packages:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "version unavailable")
        print(f"{package:<12} OK   {version}")
    except ImportError:
        print(f"{package:<12} NOT INSTALLED")

pandas       OK   3.0.5
numpy        OK   2.4.6
matplotlib   OK   3.11.1
sklearn      OK   1.9.0
imblearn     OK   0.14.2
xgboost      OK   3.4.0


## 1. Load Raw SECOM Data

The original SECOM dataset is provided as two separate files:

- `secom.data`: anonymized semiconductor process sensor measurements
- `secom_labels.data`: Pass/Fail labels and timestamps

In this step, both files are loaded separately and their structures are verified before merging.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
DATA_DIR = Path("../data/raw")

SECOM_DATA_PATH = DATA_DIR / "secom.data"
SECOM_LABEL_PATH = DATA_DIR / "secom_labels.data"

print("Sensor data:", SECOM_DATA_PATH)
print("Label data :", SECOM_LABEL_PATH)

Sensor data: ..\data\raw\secom.data
Label data : ..\data\raw\secom_labels.data


In [5]:
print("secom.data exists:", SECOM_DATA_PATH.exists())
print("secom_labels.data exists:", SECOM_LABEL_PATH.exists())

secom.data exists: True
secom_labels.data exists: True


In [6]:
sensor_df = pd.read_csv(
    SECOM_DATA_PATH,
    sep=r"\s+",
    header=None
)

sensor_df.head()

,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [7]:
sensor_df.columns = [
    f"feature_{i}"
    for i in range(sensor_df.shape[1])
]

sensor_df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_580,feature_581,feature_582,feature_583,feature_584,feature_585,feature_586,feature_587,feature_588,feature_589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [8]:
label_df = pd.read_csv(
    SECOM_LABEL_PATH,
    sep=r"\s+",
    header=None
)

label_df.head()

,0,1
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [9]:
print("Label data shape:", label_df.shape)

Label data shape: (1567, 2)


In [10]:
label_df.head(10)

,0,1
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00
5,-1,19/07/2008 17:53:00
6,-1,19/07/2008 19:44:00
7,-1,19/07/2008 19:45:00
8,-1,19/07/2008 20:24:00
9,-1,19/07/2008 21:35:00


## 2. Prepare Labels and Timestamps

The label file contains two columns:

- `Pass/Fail`: manufacturing result (`-1 = Pass`, `1 = Fail`)
- `Time`: timestamp of the observation

Before merging the label information with the sensor measurements,
the columns are renamed and the timestamp is converted to datetime format.

In [11]:
label_df.columns = ["Pass/Fail", "Time"]

label_df.head()

,Pass/Fail,Time
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [12]:
label_df.dtypes

Pass/Fail    int64
Time           str
dtype: object

In [13]:
label_df["Time"] = pd.to_datetime(
    label_df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

label_df.dtypes

Pass/Fail             int64
Time         datetime64[us]
dtype: object

## 3. Validate Data Alignment

Before merging the sensor measurements and labels,
the number of observations must be identical.

A mismatch could result in incorrect labels being assigned to sensor records.

In [14]:
print("Sensor rows:", len(sensor_df))
print("Label rows :", len(label_df))

assert len(sensor_df) == len(label_df), \
    "Sensor data and label data have different numbers of rows."

print("Row count validation passed.")

Sensor rows: 1567
Label rows : 1567
Row count validation passed.


In [15]:
print("Sensor dataset shape:", sensor_df.shape)

Sensor dataset shape: (1567, 590)


In [16]:
secom_df = sensor_df.copy()

secom_df["Time"] = label_df["Time"]
secom_df["Pass/Fail"] = label_df["Pass/Fail"]

secom_df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_582,feature_583,feature_584,feature_585,feature_586,feature_587,feature_588,feature_589,Time,Pass/Fail
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,2008-07-19 11:55:00,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,2008-07-19 12:32:00,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,2008-07-19 13:17:00,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,2008-07-19 14:43:00,-1
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,2008-07-19 15:22:00,-1


In [17]:
print("Final SECOM dataset shape:", secom_df.shape)

Final SECOM dataset shape: (1567, 592)


In [18]:
print("Number of samples :", secom_df.shape[0])
print("Sensor features   :", sensor_df.shape[1])
print("Total columns     :", secom_df.shape[1])

print("\nTarget values:")
print(secom_df["Pass/Fail"].unique())

Number of samples : 1567
Sensor features   : 590
Total columns     : 592

Target values:
[-1  1]


## 4. Inspect Pass/Fail Distribution

The SECOM dataset is highly imbalanced.

Since failed semiconductor samples are much rarer than passed samples,
class imbalance must be considered when selecting evaluation metrics
and modeling strategies.

In [19]:
target_counts = secom_df["Pass/Fail"].value_counts().sort_index()

target_counts

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

In [20]:
target_ratio = (
    secom_df["Pass/Fail"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

target_ratio

Pass/Fail
-1    93.363114
 1     6.636886
Name: proportion, dtype: float64

93개 = Pass
7개  = Fail

In [21]:
pass_count = (secom_df["Pass/Fail"] == -1).sum()
fail_count = (secom_df["Pass/Fail"] == 1).sum()
total_count = len(secom_df)

yield_rate = pass_count / total_count * 100
fail_rate = fail_count / total_count * 100

print(f"Total samples : {total_count}")
print(f"Pass samples  : {pass_count}")
print(f"Fail samples  : {fail_count}")
print(f"Yield rate    : {yield_rate:.2f}%")
print(f"Fail rate     : {fail_rate:.2f}%")

Total samples : 1567
Pass samples  : 1463
Fail samples  : 104
Yield rate    : 93.36%
Fail rate     : 6.64%


## 5. Save Integrated Dataset

The original sensor measurements and label file are combined into a single
dataset for subsequent exploratory analysis and modeling.

The processed dataset is stored locally and excluded from Git tracking.

In [22]:
OUTPUT_PATH = Path("../data/processed/uci-secom.csv")

secom_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved to:", OUTPUT_PATH)
print("File exists:", OUTPUT_PATH.exists())

Saved to: ..\data\processed\uci-secom.csv
File exists: True


## 6. Basic Data Quality Check

Before preprocessing, the dataset is inspected for:

- data types
- missing values
- duplicated observations
- constant features
- high-missing-rate features

These checks identify potential data-quality issues that should be addressed
in later preprocessing steps.

In [23]:
secom_df.dtypes.value_counts()

float64           590
datetime64[us]      1
int64               1
Name: count, dtype: int64

In [24]:
print("Dataset shape:", secom_df.shape)
print("\nData type distribution:")
print(secom_df.dtypes.value_counts())

Dataset shape: (1567, 592)

Data type distribution:
float64           590
datetime64[us]      1
int64               1
Name: count, dtype: int64


In [25]:
total_missing = secom_df.isna().sum().sum()
total_values = secom_df.size

overall_missing_rate = total_missing / total_values * 100

print(f"Total missing values : {total_missing:,}")
print(f"Total data points    : {total_values:,}")
print(f"Overall missing rate : {overall_missing_rate:.2f}%")

Total missing values : 41,951
Total data points    : 927,664
Overall missing rate : 4.52%


In [26]:
feature_cols = [
    col for col in secom_df.columns
    if col.startswith("feature_")
]

missing_summary = pd.DataFrame({
    "missing_count": secom_df[feature_cols].isna().sum(),
    "missing_ratio": secom_df[feature_cols].isna().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "missing_ratio",
    ascending=False
)

missing_summary.head(20)

,missing_count,missing_ratio
feature_292,1429,91.193363
feature_293,1429,91.193363
feature_158,1429,91.193363
feature_157,1429,91.193363
feature_492,1341,85.577537
feature_85,1341,85.577537
feature_358,1341,85.577537
feature_220,1341,85.577537
feature_244,1018,64.964901
feature_517,1018,64.964901


In [27]:
features_with_missing = (missing_summary["missing_count"] > 0).sum()

print("Total sensor features :", len(feature_cols))
print("Features with missing :", features_with_missing)
print(
    "Features without missing:",
    len(feature_cols) - features_with_missing
)

Total sensor features : 590
Features with missing : 538
Features without missing: 52


### Duplicate Observations

Duplicated observations are checked before modeling to prevent repeated records
from unintentionally influencing model training and evaluation.

In [28]:
duplicate_count = secom_df.duplicated().sum()

print("Duplicated rows:", duplicate_count)

Duplicated rows: 0


In [29]:
sensor_duplicate_count = secom_df[feature_cols].duplicated().sum()

print(
    "Duplicated sensor observations:",
    sensor_duplicate_count
)

Duplicated sensor observations: 0


In [30]:
unique_counts = secom_df[feature_cols].nunique(
    dropna=True
)

constant_features = unique_counts[
    unique_counts <= 1
].index.tolist()

print(
    "Number of constant features:",
    len(constant_features)
)

print("\nConstant features:")
print(constant_features)

Number of constant features: 116

Constant features:
['feature_5', 'feature_13', 'feature_42', 'feature_49', 'feature_52', 'feature_69', 'feature_97', 'feature_141', 'feature_149', 'feature_178', 'feature_179', 'feature_186', 'feature_189', 'feature_190', 'feature_191', 'feature_192', 'feature_193', 'feature_194', 'feature_226', 'feature_229', 'feature_230', 'feature_231', 'feature_232', 'feature_233', 'feature_234', 'feature_235', 'feature_236', 'feature_237', 'feature_240', 'feature_241', 'feature_242', 'feature_243', 'feature_256', 'feature_257', 'feature_258', 'feature_259', 'feature_260', 'feature_261', 'feature_262', 'feature_263', 'feature_264', 'feature_265', 'feature_266', 'feature_276', 'feature_284', 'feature_313', 'feature_314', 'feature_315', 'feature_322', 'feature_325', 'feature_326', 'feature_327', 'feature_328', 'feature_329', 'feature_330', 'feature_364', 'feature_369', 'feature_370', 'feature_371', 'feature_372', 'feature_373', 'feature_374', 'feature_375', 'feature_

In [31]:
numeric_features = secom_df[feature_cols]

positive_inf = np.isposinf(
    numeric_features.to_numpy()
).sum()

negative_inf = np.isneginf(
    numeric_features.to_numpy()
).sum()

print("Positive infinity values:", positive_inf)
print("Negative infinity values:", negative_inf)

Positive infinity values: 0
Negative infinity values: 0


In [32]:
print(
    "Missing Pass/Fail labels:",
    secom_df["Pass/Fail"].isna().sum()
)

print(
    "Missing timestamps:",
    secom_df["Time"].isna().sum()
)

Missing Pass/Fail labels: 0
Missing timestamps: 0


In [33]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Number of samples",
        "Number of sensor features",
        "Total columns",
        "Pass samples",
        "Fail samples",
        "Yield rate (%)",
        "Fail rate (%)",
        "Features with missing values",
        "Constant features",
        "Duplicated rows"
    ],
    "Value": [
        len(secom_df),
        len(feature_cols),
        secom_df.shape[1],
        pass_count,
        fail_count,
        round(yield_rate, 2),
        round(fail_rate, 2),
        features_with_missing,
        len(constant_features),
        duplicate_count
    ]
})

quality_summary

,Metric,Value
0,Number of samples,1567.00
1,Number of sensor features,590.00
2,Total columns,592.00
3,Pass samples,1463.00
4,Fail samples,104.00
5,Yield rate (%),93.36
6,Fail rate (%),6.64
7,Features with missing values,538.00
8,Constant features,116.00
9,Duplicated rows,0.00


In [34]:
SUMMARY_PATH = Path(
    "../results/metrics/data_quality_summary.csv"
)

quality_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("Saved to:", SUMMARY_PATH)
print("File exists:", SUMMARY_PATH.exists())

Saved to: ..\results\metrics\data_quality_summary.csv
File exists: True


## 7. Initial Findings

The initial inspection of the SECOM dataset identified three major challenges
for semiconductor yield prediction:

1. **Severe class imbalance**  
   Fail samples represent only a small fraction of the dataset.

2. **Missing sensor measurements**  
   Several process features contain missing observations and require
   appropriate preprocessing.

3. **High dimensionality**  
   The dataset contains 590 anonymized process variables relative to only
   1,567 observations.

These issues will be addressed in subsequent notebooks through careful
preprocessing, class-imbalance handling, feature selection, and model
evaluation.

In [35]:
# Features containing only missing values
all_missing_features = [
    col for col in feature_cols
    if secom_df[col].isna().all()
]

# Features containing exactly one unique non-missing value
single_value_features = [
    col for col in feature_cols
    if secom_df[col].nunique(dropna=True) == 1
]

print("All-missing features :", len(all_missing_features))
print("Single-value features:", len(single_value_features))

print("\nAll-missing feature names:")
print(all_missing_features)

print("\nSingle-value feature names:")
print(single_value_features)

All-missing features : 0
Single-value features: 116

All-missing feature names:
[]

Single-value feature names:
['feature_5', 'feature_13', 'feature_42', 'feature_49', 'feature_52', 'feature_69', 'feature_97', 'feature_141', 'feature_149', 'feature_178', 'feature_179', 'feature_186', 'feature_189', 'feature_190', 'feature_191', 'feature_192', 'feature_193', 'feature_194', 'feature_226', 'feature_229', 'feature_230', 'feature_231', 'feature_232', 'feature_233', 'feature_234', 'feature_235', 'feature_236', 'feature_237', 'feature_240', 'feature_241', 'feature_242', 'feature_243', 'feature_256', 'feature_257', 'feature_258', 'feature_259', 'feature_260', 'feature_261', 'feature_262', 'feature_263', 'feature_264', 'feature_265', 'feature_266', 'feature_276', 'feature_284', 'feature_313', 'feature_314', 'feature_315', 'feature_322', 'feature_325', 'feature_326', 'feature_327', 'feature_328', 'feature_329', 'feature_330', 'feature_364', 'feature_369', 'feature_370', 'feature_371', 'feature_3

In [36]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Number of samples",
        "Number of sensor features",
        "Total columns",
        "Pass samples",
        "Fail samples",
        "Yield rate (%)",
        "Fail rate (%)",
        "Features with missing values",
        "All-missing features",
        "Single-value features",
        "Duplicated rows"
    ],
    "Value": [
        len(secom_df),
        len(feature_cols),
        secom_df.shape[1],
        pass_count,
        fail_count,
        round(yield_rate, 2),
        round(fail_rate, 2),
        features_with_missing,
        len(all_missing_features),
        len(single_value_features),
        duplicate_count
    ]
})

quality_summary

,Metric,Value
0,Number of samples,1567.00
1,Number of sensor features,590.00
2,Total columns,592.00
3,Pass samples,1463.00
4,Fail samples,104.00
5,Yield rate (%),93.36
6,Fail rate (%),6.64
7,Features with missing values,538.00
8,All-missing features,0.00
9,Single-value features,116.00


In [37]:
SUMMARY_PATH = Path(
    "../results/metrics/data_quality_summary.csv"
)

quality_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("Saved to:", SUMMARY_PATH)

Saved to: ..\results\metrics\data_quality_summary.csv
